# Detección de Outliers vs. Eventos Reales
### IQR, Z-score, residual STL e Isolation Forest

`Fase 1 · Video 6` — serie **Forecasting de Ventas de la A a la Z**

En el **Video 5** cerramos el análisis de la estructura temporal. Aquí atacamos los **valores atípicos**
con una ventaja única: como generamos el dataset, sabemos qué picos fueron eventos reales, así que
medimos de verdad qué método los detecta (precision/recall).

---
## 1. Librerías y carga de datos

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from statsmodels.tsa.seasonal import STL
from sklearn.ensemble import IsolationForest
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor': '#F8FAFC', 'axes.facecolor': '#F8FAFC',
    'axes.grid': True, 'grid.color': '#E2E8F0',
    'grid.linewidth': 0.8, 'font.size': 11,
})
BLUE, RED, GREEN, PURPLE, ORANGE = '#2563EB','#DC2626','#16A34A','#7C3AED','#EA580C'
money_fmt = FuncFormatter(lambda x, _: f'${x/1e6:.1f}M')
print('✅ Librerías cargadas')

In [ ]:
def find_csv(filename='sales_history.csv', max_levels=4):
    base = Path().resolve()
    for _ in range(max_levels):
        candidate = base / 'output' / filename
        if candidate.exists():
            return candidate
        base = base.parent
    raise FileNotFoundError(f"No se encontró '{filename}'. Corre main.py primero.")

csv_path = find_csv()
df_raw   = pd.read_csv(csv_path, nrows=2)
date_col = next(c for c in df_raw.columns if 'date' in c.strip().lower())
df = pd.read_csv(csv_path, parse_dates=[date_col])
df.rename(columns={date_col: 'date'}, inplace=True)
df.sort_values('date', inplace=True)

# Serie diaria de revenue total (el evento está codificado por día)
daily = df.groupby('date')['revenue'].sum().sort_index()
daily.index = pd.to_datetime(daily.index)

event_by_date = df.groupby('date')['event'].first()
event_by_date.index = pd.to_datetime(event_by_date.index)
is_event = (event_by_date != 'Regular').reindex(daily.index).fillna(False)

print(f'✅ Serie diaria: {len(daily)} días | {daily.index.min().date()} → {daily.index.max().date()}')
print(f'   Días de evento real (verdad de campo): {int(is_event.sum())} '
      f'({is_event.mean():.1%} del total)')
print(f'   Eventos presentes: {sorted(set(event_by_date) - {"Regular"})}')

---
## 2. Tipos de outlier en una serie de tiempo

No todos los "puntos raros" son iguales. La distinción cambia cómo los detectas y qué haces con ellos:

| Tipo | Qué es | Ejemplo en ventas |
|---|---|---|
| **Aditivo (AO)** | Un punto aislado fuera de rango | Un día con un pico de un solo evento |
| **Sostenido / cambio de nivel** | Varios puntos consecutivos altos/bajos | Una semana completa de Buen Fin |
| **Innovacional** | Un shock que se propaga por la autocorrelación | Un quiebre de stock que arrastra días siguientes |
| **Estacional aparente** | Alto pero **esperable** por la temporada | Diciembre siempre vende más — *no* es outlier |

La trampa central: en una serie con **tendencia y estacionalidad**, un valor alto puede ser
perfectamente normal para su fecha. Por eso los métodos que ignoran el tiempo (IQR, Z-score global)
fallan — y los que primero quitan tendencia + estacionalidad (residual STL) aciertan.

---
## 3. Métodos ingenuos: IQR y Z-score

Los dos sospechosos habituales, aplicados **directo sobre el revenue**:

- **IQR:** marca como outlier todo lo que supere `Q3 + 1.5·IQR`.
- **Z-score:** marca lo que esté a más de 3 desviaciones de la media global.

Ambos asumen que los datos son **estacionarios e idénticamente distribuidos** — justo lo que una serie
con tendencia y estacionalidad **no** es. Veamos qué tan bien (o mal) les va.

In [ ]:
q1, q3 = daily.quantile([0.25, 0.75])
iqr    = q3 - q1
upper  = q3 + 1.5 * iqr
iqr_out = daily.index[daily > upper]

z = (daily - daily.mean()) / daily.std()
z_out = daily.index[z.abs() > 3]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for ax, out_idx, title, thr_line in [
    (axes[0], iqr_out, f'IQR — marca {len(iqr_out)} días (fence = Q3 + 1.5·IQR)', upper),
    (axes[1], z_out,   f'Z-score global — marca {len(z_out)} días (|z| > 3)',     None),
]:
    ax.plot(daily.index, daily.values, color=BLUE, linewidth=0.8, alpha=0.7)
    ax.scatter(out_idx, daily.loc[out_idx], color=RED, s=25, zorder=3, label='Marcado outlier')
    if thr_line is not None:
        ax.axhline(thr_line, color=ORANGE, linestyle='--', linewidth=1, label='Fence IQR')
    ax.yaxis.set_major_formatter(money_fmt)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(loc='upper left')
fig.suptitle('Métodos ingenuos sobre el revenue crudo', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

def hit_rate(idx):
    if len(idx) == 0:
        return 0.0
    return float(is_event.loc[idx].mean())

print(f'IQR      : {len(iqr_out):>3} marcados, {hit_rate(iqr_out):.0%} caen en un evento real')
print(f'Z-score  : {len(z_out):>3} marcados, {hit_rate(z_out):.0%} caen en un evento real')
print('\nSorpresa: en ESTE dataset los eventos son picos multiplicativos enormes y sincronizados')
print('(todos los SKUs a la vez), así que dominan la cola alta y hasta el IQR los detecta bien.')
print('Pero es frágil y dependiente del umbral: |z|>3 solo atrapa los eventos más grandes y se')
print('pierde los suaves (San Valentín, Reyes) — lo veremos como recall bajo en la sección 8.')

---
## 4. Lo correcto: Z-score robusto sobre el residual STL

La idea: **primero quitamos lo explicable** (tendencia + estacionalidad con STL, como en el Video 4) y
buscamos outliers en lo que queda — el **residual**. Un día es anómalo si su residual es grande *para su
propia fecha*, no comparado con toda la serie.

Y usamos una versión **robusta** del Z-score, basada en la **mediana** y la **MAD** (desviación absoluta
mediana) en lugar de media y desviación estándar — porque la media y la std ya están contaminadas por los
propios outliers que queremos detectar:

$$z_{\text{rob}} = \frac{r_t - \text{mediana}(r)}{1.4826 \cdot \text{MAD}(r)}$$

El factor 1.4826 hace que la MAD sea comparable a la desviación estándar bajo normalidad.

In [ ]:
# STL sobre log1p (estabiliza varianza); period=7 elimina el patrón de día de semana.
# La tendencia LOESS es suave: NO absorbe los picos cortos de evento → quedan en el residual.
res_stl = STL(np.log1p(daily), period=7, robust=True).fit()
resid   = res_stl.resid

med = resid.median()
mad = (resid - med).abs().median()
robust_z = (resid - med) / (1.4826 * mad)

THR = 3.5
stl_out = daily.index[robust_z.abs() > THR]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(daily.index, daily.values, color=BLUE, linewidth=0.8, alpha=0.7)
axes[0].scatter(stl_out, daily.loc[stl_out], color=RED, s=28, zorder=3, label='Outlier (STL)')
axes[0].yaxis.set_major_formatter(money_fmt)
axes[0].set_title(f'Serie diaria — {len(stl_out)} outliers por residual STL', fontsize=12, fontweight='bold')
axes[0].legend(loc='upper left')

axes[1].plot(resid.index, robust_z.values, color=PURPLE, linewidth=0.8)
axes[1].axhline( THR, color=RED, linestyle='--', linewidth=1, label=f'±{THR} (umbral)')
axes[1].axhline(-THR, color=RED, linestyle='--', linewidth=1)
axes[1].axhline(0, color='black', linewidth=0.6)
axes[1].scatter(stl_out, robust_z.loc[stl_out], color=RED, s=20, zorder=3)
axes[1].set_title('Z-score robusto del residual STL', fontsize=12, fontweight='bold')
axes[1].legend(loc='upper left')

fig.suptitle('Detección sobre el residual (tendencia + estacionalidad removidas)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'STL robusto: {len(stl_out):>3} marcados, {is_event.loc[stl_out].mean():.0%} caen en un evento real')
print('Al mirar el RESIDUAL (no el nivel), recupera también los eventos SUAVES que el Z-score')
print('global se pierde → mayor recall. El precio: marca también días de puro ruido alto, así')
print('que su precisión baja. Es el clásico trade-off precision/recall (lo medimos en la sección 8).')

---
## 5. Isolation Forest — detección multivariante

Los métodos anteriores miran **una sola variable**. Isolation Forest (Liu et al., 2008) es no
supervisado y multivariante: construye árboles que **aíslan** puntos con cortes aleatorios. La intuición
es elegante — **un outlier se aísla con pocos cortes** porque está en una región poco poblada del espacio
de features; un punto normal necesita muchos cortes.

Le damos varias señales por día (nivel, día de semana, mes, desviación local, residual STL) y le pedimos
que marque el ~5% más anómalo (`contamination`).

In [ ]:
features = pd.DataFrame({
    'revenue':   daily.values,
    'log_rev':   np.log1p(daily.values),
    'dow':       daily.index.dayofweek,
    'month':     daily.index.month,
    'local_dev': (daily - daily.rolling(7, center=True, min_periods=1).median()).values,
    'stl_resid': resid.values,
}, index=daily.index)

iso = IsolationForest(contamination=0.05, random_state=42, n_estimators=300)
labels = iso.fit_predict(features)          # -1 = anómalo, 1 = normal
iso_out = daily.index[labels == -1]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily.index, daily.values, color=BLUE, linewidth=0.8, alpha=0.7)
ax.scatter(iso_out, daily.loc[iso_out], color=RED, s=28, zorder=3, label='Anomalía (Isolation Forest)')
ax.yaxis.set_major_formatter(money_fmt)
ax.set_title(f'Isolation Forest — {len(iso_out)} días marcados (contamination=5%)',
             fontsize=12, fontweight='bold')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

print(f'Isolation Forest: {len(iso_out):>3} marcados, {is_event.loc[iso_out].mean():.0%} son eventos reales')

---
## 6. Comparación de los métodos

Superponemos qué marca cada uno. Fíjate **dónde** difieren: los métodos ingenuos se concentran en el
tramo final (más nivel), mientras que los basados en residual/features se reparten por los picos.

In [ ]:
methods = {
    'IQR':               iqr_out,
    'Z-score':           z_out,
    'STL robusto':       stl_out,
    'Isolation Forest':  iso_out,
}
colors = {'IQR': ORANGE, 'Z-score': PURPLE, 'STL robusto': GREEN, 'Isolation Forest': RED}

fig, axes = plt.subplots(len(methods), 1, figsize=(14, 10), sharex=True)
for ax, (name, idx) in zip(axes, methods.items()):
    ax.plot(daily.index, daily.values, color=BLUE, linewidth=0.7, alpha=0.6)
    ax.scatter(idx, daily.loc[idx], color=colors[name], s=18, zorder=3)
    ax.scatter(daily.index[is_event], daily.loc[is_event], color='#CBD5E1', s=6, zorder=1)
    ax.yaxis.set_major_formatter(money_fmt)
    ax.set_title(f'{name} — {len(idx)} marcados', fontsize=11, fontweight='bold', loc='left')
fig.suptitle('Qué marca cada método (gris = eventos reales de fondo)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. La revelación: cómo se generaron los eventos

Aquí está lo que ningún dataset público te da. Estos son los eventos que **inyectamos** al generar el
dataset, con su multiplicador de demanda — directo del código fuente (`src/events.py`):

In [ ]:
# Importamos la definición REAL de eventos desde el código del generador
sys.path.insert(0, str(csv_path.parents[1]))   # raíz del repo (output/ cuelga de ahí)
from src.events import SPECIAL_EVENTS

ev_table = pd.DataFrame([
    {'evento': e.name, 'mes': e.month, 'días': f'{e.day_start}–{e.day_end}', 'multiplicador': e.mult}
    for e in sorted(SPECIAL_EVENTS, key=lambda e: e.mult, reverse=True)
])
print('Eventos inyectados en el dataset (verdad de campo):')
print(ev_table.to_string(index=False))
print()
print('Nota: multiplicadores altos (Buen Fin ×3.5, Black Fri ×3.0) son fáciles de detectar;')
print('los suaves (San Valentín ×1.5, Reyes ×1.6) se confunden con ruido — ahí falla el recall.')

### ¿Qué método recupera mejor la verdad?

Con la columna `event` como *ground truth* podemos calcular métricas reales de clasificación:

- **Precision** = de lo que marqué como outlier, ¿qué fracción era evento real? (evita falsas alarmas)
- **Recall** = de los eventos reales, ¿qué fracción detecté? (evita perderme eventos)
- **F1** = balance entre ambas.

In [ ]:
def scores(flagged_idx, truth):
    flagged = pd.Series(False, index=truth.index)
    flagged.loc[flagged_idx] = True
    tp = int((flagged & truth).sum())
    fp = int((flagged & ~truth).sum())
    fn = int((~flagged & truth).sum())
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / (tp + fn) if (tp + fn) else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return {'marcados': tp + fp, 'precision': prec, 'recall': rec, 'f1': f1}

report = pd.DataFrame({name: scores(idx, is_event) for name, idx in methods.items()}).T
report = report[['marcados', 'precision', 'recall', 'f1']]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(report))
w = 0.27
ax.bar(x - w, report['precision'], w, label='Precision', color=BLUE)
ax.bar(x,     report['recall'],    w, label='Recall',    color=GREEN)
ax.bar(x + w, report['f1'],        w, label='F1',        color=ORANGE)
ax.set_xticks(x)
ax.set_xticklabels(report.index)
ax.set_ylim(0, 1)
ax.set_title('Detección de eventos reales — precision / recall / F1', fontsize=12, fontweight='bold')
ax.legend()
for i, f1 in enumerate(report['f1']):
    ax.text(i + w, f1 + 0.02, f'{f1:.2f}', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

print(report.to_string(float_format=lambda v: f'{v:.3f}'))
best_f1   = report['f1'].idxmax()
best_prec = report['precision'].idxmax()
best_rec  = report['recall'].idxmax()
print(f'\n→ Mejor F1: {best_f1}  |  mejor precision: {best_prec}  |  mejor recall: {best_rec}')
print('  Lección incómoda pero valiosa: aquí el método MÁS simple (IQR) gana. Los eventos son picos')
print('  multiplicativos tan extremos que dominan la distribución, y sofisticar no ayuda. Moraleja:')
print('  MIDE contra la verdad antes de asumir que el método "avanzado" es mejor.')
print('  Y observa el trade-off: el Z-score da precision perfecta pero se pierde ~60% de los eventos;')
print('  el residual STL recupera esos eventos suaves, pero a costa de más falsas alarmas.')

---
## 9. Conclusiones y puente al Video 7

| Método | Perfil en este dataset | Cuándo brilla |
|---|---|---|
| IQR | Mejor balance aquí (F1 y recall altos) | Anomalías extremas que dominan la cola |
| Z-score global | Precision perfecta, recall bajo | Solo quieres los picos más grandes, sin falsas alarmas |
| **Z robusto sobre residual STL** | Recupera eventos suaves (mejor recall que Z-score) | La estacionalidad domina y hay que ver *bajo* ella |
| **Isolation Forest** | Precision alta, multivariante | Muchas señales y costo de falsa alarma elevado |

### Las reglas que te llevas

1. **Mide, no asumas.** El método más simple (IQR) ganó aquí en F1 — sofisticar no siempre ayuda. Sin la
   verdad de campo lo habríamos "adivinado" al revés. Valida antes de decidir.
2. **Es un trade-off precision/recall.** Elige por costo de negocio: ¿duele más una falsa alarma
   (prioriza precision → Z-score / Isolation Forest) o perderte un evento (prioriza recall)?
3. **Detecta sobre el residual cuando la estacionalidad pese más que las anomalías.** Aquí los eventos
   eran tan extremos que el nivel bastó; en series donde la temporada domina, mirar el residual es lo que
   evita confundir "alto por diciembre" con "alto de verdad".
4. **Usa estadísticos robustos (mediana, MAD).** La media y la std ya están contaminadas por los outliers.
5. **Un outlier no siempre se borra.** Si es un evento conocido, se **modela** como variable exógena
   (Videos 8 y 12); solo el ruido/errores se limpia.
6. **Si tienes la verdad, úsala.** Precision/recall convierten "marcar puntos raros" en una evaluación
   objetiva — el lujo que este dataset sintético te regala.

### Por qué esto importa para lo que viene

El tratamiento de outliers **no es uniforme entre SKUs**: un SKU de alta rotación con picos promocionales
se modela distinto a uno de demanda intermitente lleno de ceros (que un detector de outliers marcaría mal).
Eso nos lleva directo al siguiente video.

---

### Próximo video
**Video 7 — Diagnóstico de negocio: ¿qué SKUs merecen un modelo dedicado?**
Segmentaremos el portafolio en *tiers* de forecasting (coeficiente de variación, intermitencia,
contribución al revenue y predictibilidad) para decidir dónde invertir un SARIMAX dedicado, dónde un
XGBoost global y dónde un método de demanda intermitente (Croston, → Video 14).